# Fault-labelled ML training data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_fault_ml_dataset.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 3.5 Fault-Detection Training Pairs
2. 10.1 Capstone: Fault-Detection Dataset
3. 9.5 Export Your Model

In [ ]:
# Ask Ogbon starter: Generate ML training data
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
N_SAMPLES = 500     # interview asked for 2000; raise this for the full run
NX = NZ = 128
F_HZ = 28.0
MAX_THROW = 12
SEED = 7
LABEL = 'fault'

In [ ]:
def ricker(f, dt, n):
    t = (np.arange(n) - n//2) * dt
    a = (np.pi * f * t) ** 2
    return (1 - 2*a) * np.exp(-a)

rng = np.random.default_rng(SEED)
w = ricker(F_HZ, 0.002, 61)
X = np.zeros((N_SAMPLES, NZ, NX), dtype=np.float32)
Y = np.zeros((N_SAMPLES, NZ, NX), dtype=np.float32)

for i in range(N_SAMPLES):
    n_layers = rng.integers(4, 9)
    tops = np.sort(rng.integers(5, NZ - 5, n_layers))
    imps = rng.uniform(4000, 9000, n_layers + 1)
    fx = rng.integers(NX // 4, 3 * NX // 4)
    throw = rng.integers(3, MAX_THROW + 1)
    dip = rng.uniform(-0.3, 0.3)
    ai = np.full((NZ, NX), imps[0], dtype=np.float32)
    for k, top in enumerate(tops):
        for ix in range(NX):
            t = top + (throw if ix >= fx + dip * (top - NZ // 2) else 0)
            ai[int(np.clip(t, 0, NZ - 1)):, ix] = imps[k + 1]
    rc = np.zeros_like(ai)
    rc[1:, :] = (ai[1:, :] - ai[:-1, :]) / (ai[1:, :] + ai[:-1, :])
    sec = np.apply_along_axis(lambda c: np.convolve(c, w, 'same'), 0, rc)
    X[i] = sec
    if LABEL == 'fault':
        for iz in range(NZ):
            xi = int(np.clip(fx + dip * (iz - NZ // 2), 0, NX - 1))
            Y[i, iz, xi] = 1.0
    elif LABEL == 'horizon':
        for k, top in enumerate(tops):
            for ix in range(NX):
                t = top + (throw if ix >= fx + dip * (top - NZ // 2) else 0)
                Y[i, int(np.clip(t, 0, NZ - 1)), ix] = 1.0
    else:  # facies: the layer-index grid itself is the label
        Y[i] = np.digitize(ai, np.sort(imps))

np.savez('ssm_training_pairs.npz', X=X, Y=Y)
print('saved', X.shape, '->', 'ssm_training_pairs.npz')

fig, ax = plt.subplots(2, 4, figsize=(10, 5))
for j in range(4):
    ax[0, j].imshow(X[j], cmap='gray', aspect='auto')
    ax[1, j].imshow(Y[j], cmap='hot', aspect='auto')
    ax[0, j].set_axis_off(); ax[1, j].set_axis_off()
ax[0, 0].set_title('section'); ax[1, 0].set_title('label')
plt.tight_layout(); plt.show()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)